In [ ]:
import bacco
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append("/cosmos_storage/home/fgmaion/MTNG-resims/src")
import utils

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
## Load the Zooms
sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME

name_list = ['LH_{:d}'.format(i) for i in range(2)] + ['fiducial']

snap = 264
zoom = {}

loaded = []
for i in range(len(name_list)):
    if i<2:
        base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/LH_{:d}/hydro_output/".format(i)
    else:
        base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/fiducial/hydro_output/"
    zoom[name_list[i]] = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(snap,snap), sim_format='TNG500', fixedPk=True, use_orphans=False,\
                            tau=tau, ns=ns, sigma8=sigma8, dm_file="snapdir_{:03d}/snapshot_{:03d}".format(snap,snap), use_ids=True, numpart=4320)

# Load the Halo Selection
with open("/cosmos_storage/simulations/TNG_Family/MN5_resims/resims_info/hydro_halo_sel_1pmbin.txt") as f:
    final_sel = []

    for line in f.readlines():
        final_sel.append(int(line.split()[0]))

final_sel = np.array(final_sel)

# Perform the cross-match with MTNG halos
xmatch = {}

for i in range(len(name_list)):
    xmatch[name_list[i]] = utils.cross_match(zoom[name_list[i]], snap=264, name=name_list[i])

# Load MTNG and get the fraction of halos to do the upweighting
mtng = bacco.utils.load_MTNG(adr="/cosmos_storage/simulations/TNG_Family/MTNG/", snap=264)

m200b = np.log10(1e10 * mtng.fof['halo_m200b'])

mbins = np.concatenate(
    (np.arange(11, 11.5, 0.0025),
    np.arange(11.5, 12.5, 0.005),
    np.arange(12.5, 13.5, 0.025),
    np.arange(13.5, 15.01, 0.125))
)

h_frac = np.zeros(len(final_sel))
for m in range(len(mbins)-1):
    h_frac[m] = np.where(( m200b[final_sel]>=mbins[m]) & ( m200b[final_sel]<mbins[m+1]))[0].shape[0] / \
             np.where(( m200b>=mbins[m]) & ( m200b<mbins[m+1]))[0].shape[0]

zoom_split = {}
zoom_sel = {}
for i in range(len(name_list)):
    zoom_split[name_list[i]] = utils.split_halos(zoom[name_list[i]])

    zoom_sel[name_list[i]] = {}

    zoom_sel[name_list[i]]['sel'] = xmatch[name_list[i]]['ind'][:,np.newaxis,np.newaxis]
    zoom_sel[name_list[i]]['h_frac'] = h_frac[np.newaxis, :]

In [ ]:
Nbins_fgas = 15

zoom_fgas = {}
zoom_smf = {}
bh_mf = {}

for i in range(len(name_list)):
    zoom_fgas[name_list[i]] = zoom_split[name_list[i]].halo_gas_frac_v2(sel_mask=zoom_sel[name_list[i]], nbins=Nbins_fgas, draws=1)    
    zoom_smf[name_list[i]] = zoom_split[name_list[i]].halo_smf_draws(sel_mask=zoom_sel[name_list[i]], nbins=Nbins_smf, draws=1, m_30kpc=True)
    bh_mf[name_list[i]] = zoom_split[name_list[i]].bh_mf(sel_mask=zoom_sel[name_list[i]], nbins=20)

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_xscale("log")

ax.plot(zoom_fgas['fiducial']['m500c'][0], zoom_fgas['fiducial']['f_gas'][0], color='k')
ax.plot(zoom_fgas['LH_0']['m500c'][0], zoom_fgas['LH_0']['f_gas'][0], color='C0')
ax.plot(zoom_fgas['LH_1']['m500c'][0], zoom_fgas['LH_1']['f_gas'][0], color='C3')

ax.set_xlabel(r"$M_{500,c}$ [$M_\odot$]")
ax.set_ylabel(r"$f_\mathrm{gas}(<R_{500,c})$")

ax.legend(['Fiducial', 'LH_0', 'LH_1'], frameon=False)

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_xscale("log")
ax.set_yscale("log")

ax.plot(zoom_smf['fiducial']['mstar'][0], zoom_smf['fiducial']['smf'][0], color='k')
ax.plot(zoom_smf['LH_0']['mstar'][0], zoom_smf['LH_0']['smf'][0], color='C0')
ax.plot(zoom_smf['LH_1']['mstar'][0], zoom_smf['LH_1']['smf'][0], color='C3')

ax.set_xlabel(r"$M_\star$ [$M_\odot$]")
ax.set_ylabel(r"$\Phi(M_\star)$ [Mpc$^{-3}$ dex$^{-1}$]")

ax.legend(['Fiducial', 'LH_0', 'LH_1'], frameon=False)

In [ ]:
bh_mf['fiducial']['mbh']

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_xscale("log")
ax.set_yscale("log")

ax.plot(bh_mf['fiducial']['mbh'], bh_mf['fiducial']['bhmf'], color='k')
ax.plot(bh_mf['LH_0']['mbh'], bh_mf['LH_0']['bhmf'], color='C0')
ax.plot(bh_mf['LH_1']['mbh'], bh_mf['LH_1']['bhmf'], color='C3')

ax.set_xlabel(r"$M_\star$ [$M_\odot$]")
ax.set_ylabel(r"$\Phi(M_\star)$ [Mpc$^{-3}$ dex$")

ax.legend(['Fiducial', 'LH_0', 'LH_1'], frameon=False)

In [ ]:
wind_en_or      = []
wind_vel_or     = []
rho_rec_or      = []
sf_ts_or        = []
ef_kin_or       = []
ef_high_or      = []
f_re_or         = []

for i in range(31):
    if i<30:
        filename = "/cosmos_storage/simulations/TNG_Family/MN5_resims/param_LH/param_MTNG-hydro_{:d}.txt".format(i)
    else:
        filename = "/cosmos_storage/simulations/TNG_Family/MN5_resims/param_LH/param_MTNG-hydro.txt"

    with open(filename, 'r') as f:
        for line in f.readlines():
            if len(line.split())!=0:
                if line.split()[0] == 'WindEnergyIn1e51erg':
                    wind_en_or.append(float(line.split()[1]))
                if line.split()[0] == 'VariableWindVelFactor':
                    wind_vel_or.append(float(line.split()[1]))
                if line.split()[0] == 'WindFreeTravelDensFac':
                    rho_rec_or.append(float(line.split()[1]))
                if line.split()[0] == 'MaxSfrTimescale':
                    sf_ts_or.append(float(line.split()[1]))
                if line.split()[0] == 'RadioFeedbackFactor':
                    ef_kin_or.append(float(line.split()[1]))
                if line.split()[0] == 'BlackHoleFeedbackFactor':
                    ef_high_or.append(float(line.split()[1]))
                if line.split()[0] == 'RadioFeedbackReiorientationFactor':
                    f_re_or.append(float(line.split()[1]))

rho_rec_or = np.log10(rho_rec_or)
ef_kin_or = np.log10(ef_kin_or)
        
wind_en   = (np.asarray(wind_en_or) - np.mean(wind_en_or)) / np.std(wind_en_or)
wind_vel  = (np.asarray(wind_vel_or) - np.mean(wind_vel_or)) / np.std(wind_vel_or)
rho_rec   = (np.asarray(rho_rec_or) - np.mean(rho_rec_or)) / np.std(rho_rec_or)
sf_ts     = (np.asarray(sf_ts_or) - np.mean(sf_ts_or)) / np.std(sf_ts_or)
ef_kin    = (np.asarray(ef_kin_or) - np.mean(ef_kin_or)) / np.std(ef_kin_or)
ef_high   = (np.asarray(ef_high_or) - np.mean(ef_high_or)) / np.std(ef_high_or)
f_re      = (np.asarray(f_re_or) - np.mean(f_re_or)) / np.std(f_re_or)

In [ ]:
print("WindEnergyIn1e51erg:               fiducial {:.2f}, LH_0 {:.2f}, LH_1 {:.2f}".format(wind_en_or[30], wind_en_or[0], wind_en_or[1]))
print("VariableWindVelFactor:             fiducial {:.2f}, LH_0 {:.2f}, LH_1 {:.2f}".format(wind_vel_or[30], wind_vel_or[0], wind_vel_or[1] ))
print("WindFreeTravelDensFac:             fiducial {:.2f}, LH_0 {:.2f}, LH_1 {:.2f}".format(rho_rec_or[30], rho_rec_or[0], rho_rec_or[1] ))
print("MaxSfrTimescale:                   fiducial {:.4f}, LH_0 {:.4f}, LH_1 {:.4f}".format(sf_ts_or[30], sf_ts_or[0], sf_ts_or[1] ))
print("RadioFeedbackFactor:               fiducial {:.4f}, LH_0 {:.4f}, LH_1 {:.4f}".format(ef_kin_or[30], ef_kin_or[0], ef_kin_or[1] ))
print("BlackHoleFeedbackFactor:           fiducial {:.2f}, LH_0 {:.2f}, LH_1 {:.2f}".format(ef_high_or[30], ef_high_or[0], ef_high_or[1] ))
print("RadioFeedbackReiorientationFactor: fiducial {:.2f}, LH_0 {:.2f}, LH_1 {:.2f}".format(f_re_or[30], f_re_or[0], f_re_or[1] ))